
# BTS Flight Data — Initial Data Audit

## 1. Objective

This notebook explores the raw BTS Reporting Carrier On-Time Performance dataset and validates its suitability for the Flight Delay ML Platform.

The platform supports two binary classification tasks:

- **Departure Delay:** `DepDel15`
- **Arrival Delay:** `ArrDel15`

Both predictions are made 24 hours before scheduled gate departure.

## 2. Pilot Dataset

- Source: Bureau of Transportation Statistics (BTS)
- Table: Reporting Carrier On-Time Performance
- Period: January 2023
- Format: ZIP containing CSV
- Storage location: `data/raw/`

Official source:
https://transtats.bts.gov/PREZIP/

## 3. Data Engineering Principles

1. Preserve the original source files.
2. Keep raw datasets out of GitHub.
3. Validate the schema before transformation.
4. Separate raw data from processed datasets.
5. Record data quality issues before modeling.
6. Prevent current-flight outcome information from entering model features.

## 4. Initial Workflow

Download → Inspect ZIP → Verify CSV Schema → Data Audit → ETL → Parquet

The pilot uses one month of data before expanding to the full 2023–2025 dataset.


In [ ]:
from pathlib import Path
from urllib.request import Request, urlopen
import shutil
import zipfile

# Find repository root
current = Path.cwd().resolve()

ROOT = next(
    (
        path for path in [current, *current.parents]
        if (path / "pyproject.toml").exists()
    ),
    None,
)

assert ROOT is not None, "Project root not found."

RAW_DIR = ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

YEAR = 2023
MONTH = 1

filename = (
    "On_Time_Reporting_Carrier_On_Time_Performance_"
    f"1987_present_{YEAR}_{MONTH}.zip"
)

url = f"https://transtats.bts.gov/PREZIP/{filename}"
zip_path = RAW_DIR / filename

# Download only when the local ZIP is not already valid.
if not zip_path.exists() or not zipfile.is_zipfile(zip_path):
    print("Downloading:", url)

    request = Request(
        url,
        headers={"User-Agent": "Mozilla/5.0"}
    )

    temp_path = zip_path.with_suffix(".zip.part")

    try:
        with urlopen(request, timeout=90) as response:
            with temp_path.open("wb") as output:
                shutil.copyfileobj(response, output)

        if not zipfile.is_zipfile(temp_path):
            raise ValueError("Downloaded file is not a valid ZIP.")

        temp_path.replace(zip_path)

    finally:
        temp_path.unlink(missing_ok=True)

else:
    print("Valid ZIP already exists. Skipping download.")

print("\nSaved to:", zip_path)
print("File size (MB):", round(zip_path.stat().st_size / 1024**2, 2))

In [ ]:
import csv
import io
import zipfile

with zipfile.ZipFile(zip_path) as zf:
    csv_files = [
        name for name in zf.namelist()
        if name.lower().endswith(".csv")
    ]

    assert csv_files, "No CSV file found in ZIP."

    csv_name = max(
        csv_files,
        key=lambda name: zf.getinfo(name).file_size
    )

    print("CSV file:", csv_name)

    with zf.open(csv_name) as binary_file:
        with io.TextIOWrapper(
            binary_file,
            encoding="utf-8-sig",
            newline=""
        ) as text_file:

            reader = csv.DictReader(text_file)
            columns = reader.fieldnames

            print("\nTotal columns:", len(columns))
            print("\nColumn names:")
            print(columns)

            print("\nFirst 3 records:")
            for _, row in zip(range(3), reader):
                print({
                    key: row.get(key)
                    for key in [
                        "FlightDate",
                        "Reporting_Airline",
                        "Origin",
                        "Dest",
                        "DepDel15",
                        "ArrDel15",
                    ]
                })

# Check required fields
required_columns = {
    "FlightDate",
    "Reporting_Airline",
    "Flight_Number_Reporting_Airline",
    "Origin",
    "Dest",
    "CRSDepTime",
    "CRSArrTime",
    "DepDel15",
    "ArrDel15",
    "Cancelled",
    "Diverted",
}

missing = required_columns - set(columns)

assert not missing, f"Missing columns: {missing}"

print("\nInitial schema check passed!")


## 3. Initial Data Quality Audit

### Objective

Perform an initial quality assessment of the January 2023 BTS dataset before preprocessing or feature engineering.

### Audit Scope

1. Dataset dimensions and date range
2. Data types and missing values
3. Cancelled and diverted flights
4. Departure and arrival delay label distributions
5. Eligible flights for initial modeling

### Important Considerations

- The original dataset contains flights from all reporting airports.
- The modeling scope will later be restricted to six origin airports.
- `DepDel15` and `ArrDel15` must be converted to numeric labels.
- Missing labels may be associated with cancelled or diverted flights.
- Raw records should not be modified during the audit.

The audit results will guide the preprocessing and data validation pipeline.


In [ ]:
import pandas as pd
import zipfile

# Fields needed for the initial audit.
AUDIT_COLUMNS = [
    "FlightDate",
    "Reporting_Airline",
    "Flight_Number_Reporting_Airline",
    "Origin",
    "Dest",
    "CRSDepTime",
    "CRSArrTime",
    "CRSElapsedTime",
    "Distance",
    "DepDelay",
    "ArrDelay",
    "DepDel15",
    "ArrDel15",
    "Cancelled",
    "Diverted",
]

# Read only selected columns directly from the ZIP.
# zip_path was defined in the previous download cell.
with zipfile.ZipFile(zip_path) as zf:
    csv_name = next(
        name for name in zf.namelist()
        if name.lower().endswith(".csv")
    )

    with zf.open(csv_name) as file:
        df = pd.read_csv(
            file,
            usecols=AUDIT_COLUMNS,
            dtype={"Flight_Number_Reporting_Airline": "string"},
            low_memory=False,
        )

# Convert dates and numeric values.
df["FlightDate"] = pd.to_datetime(
    df["FlightDate"], errors="coerce"
)

numeric_columns = [
    "CRSDepTime",
    "CRSArrTime",
    "CRSElapsedTime",
    "Distance",
    "DepDelay",
    "ArrDelay",
    "DepDel15",
    "ArrDel15",
    "Cancelled",
    "Diverted",
]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("Dataset Shape:", df.shape)
print("Number of Rows:", len(df))
print("Number of Columns:", len(df.columns))

print("\nDate Range:")
print(df["FlightDate"].min(), "to", df["FlightDate"].max())

print("\nData Types:")
print(df.dtypes)

print("\nFirst 5 Records:")
display(df.head())

In [ ]:
# 1. Missing values
missing_report = pd.DataFrame({
    "Missing_Count": df.isna().sum(),
    "Missing_Pct": (df.isna().mean() * 100).round(2),
}).sort_values("Missing_Pct", ascending=False)

print("=== Missing Value Report ===")
display(missing_report)

# 2. Cancelled and diverted flights
print("\n=== Flight Status ===")

for col in ["Cancelled", "Diverted"]:
    print(f"\n{col}:")
    print(df[col].value_counts(dropna=False))

# 3. Restrict the modeling cohort to six origin airports.
ORIGIN_AIRPORTS = [
    "ATL", "DFW", "DEN", "ORD", "LAX", "SFO"
]

airport_mask = df["Origin"].isin(ORIGIN_AIRPORTS)

# Only completed flights with valid labels.
eligible_mask = (
    airport_mask
    & df["Cancelled"].eq(0)
    & df["Diverted"].eq(0)
    & df["DepDel15"].isin([0, 1])
    & df["ArrDel15"].isin([0, 1])
)

modeling_df = df.loc[eligible_mask].copy()

print("\n=== Modeling Cohort ===")
print("Total raw records:", len(df))
print("Six-airport records:", int(airport_mask.sum()))
print("Eligible completed flights:", len(modeling_df))

# 4. Target distributions
for target in ["DepDel15", "ArrDel15"]:
    print(f"\n=== {target} ===")

    counts = modeling_df[target].value_counts(
        dropna=False
    ).sort_index()

    print(counts)

    if len(modeling_df) > 0:
        delay_rate = modeling_df[target].mean() * 100
        print(f"Positive class rate: {delay_rate:.2f}%")
    else:
        print("No eligible records.")


## 4. Label Consistency and Missingness Audit

### Objective

Investigate missing target labels and verify that the binary delay indicators are consistent with the corresponding delay duration fields.

### Audit Questions

1. Are missing departure and arrival labels associated with cancelled or diverted flights?
2. Are `DepDel15` and `ArrDel15` consistent with the 15-minute delay definition?
3. Are there any unexpected label values?
4. Are there potential duplicate flight records?

### Validation Rules

For records with valid outcomes:

- `DepDel15 = 1` when `DepDelay >= 15`.
- `DepDel15 = 0` when `DepDelay < 15`.
- `ArrDel15 = 1` when `ArrDelay >= 15`.
- `ArrDel15 = 0` when `ArrDelay < 15`.

Missing values must not be automatically converted to zero.

Potential duplicate records will be investigated using flight identifiers rather than treating identical feature values as sufficient evidence of duplication.


In [ ]:
# Compare missing labels across flight outcomes.

for target in ["DepDel15", "ArrDel15"]:
    print(f"\n=== Missingness Audit: {target} ===")

    audit = df.assign(
        Label_Missing=df[target].isna()
    )

    status_report = (
        audit.groupby(
            ["Cancelled", "Diverted"],
            dropna=False
        )["Label_Missing"]
        .agg(
            Total_Flights="size",
            Missing_Labels="sum",
            Missing_Rate="mean"
        )
        .reset_index()
    )

    status_report["Missing_Rate"] = (
        status_report["Missing_Rate"] * 100
    ).round(2)

    display(status_report)

    # Missing labels among normal completed flights.
    normal_mask = (
        df["Cancelled"].eq(0)
        & df["Diverted"].eq(0)
    )

    unexpected_missing = (
        normal_mask & df[target].isna()
    ).sum()

    print(
        "Missing labels in completed flights:",
        unexpected_missing
    )

In [ ]:
# Verify binary labels against delay minutes.

label_pairs = [
    ("DepDelay", "DepDel15"),
    ("ArrDelay", "ArrDel15"),
]

for delay_col, label_col in label_pairs:

    print(f"\n=== Validating {label_col} ===")

    valid = df[
        df[delay_col].notna()
        & df[label_col].notna()
    ].copy()

    # Expected labels based on the 15-minute rule.
    expected = (valid[delay_col] >= 15).astype(int)

    actual = valid[label_col].astype(int)

    mismatch = expected.ne(actual)

    invalid_label = ~df[label_col].dropna().isin([0, 1])

    print("Valid records:", len(valid))
    print("Label mismatches:", int(mismatch.sum()))
    print("Invalid binary labels:", int(invalid_label.sum()))

    if mismatch.any():
        print("\nExample mismatched records:")
        display(
            valid.loc[
                mismatch,
                [
                    "FlightDate",
                    "Origin",
                    "Dest",
                    delay_col,
                    label_col,
                ]
            ].head(10)
        )

In [ ]:
# Include flight number for a more meaningful duplicate check.

flight_key_columns = [
    "FlightDate",
    "Reporting_Airline",
    "Flight_Number_Reporting_Airline",
    "Origin",
    "Dest",
    "CRSDepTime",
]

with zipfile.ZipFile(zip_path) as zf:
    with zf.open(csv_name) as file:
        key_df = pd.read_csv(
            file,
            usecols=flight_key_columns,
            dtype={"Flight_Number_Reporting_Airline": "string"},
            low_memory=False,
        )

# Count potential duplicate business keys.
duplicate_mask = key_df.duplicated(
    subset=flight_key_columns,
    keep=False,
)

print("Total records:", len(key_df))
print(
    "Records with repeated candidate keys:",
    int(duplicate_mask.sum())
)

print("\nExample repeated keys:")

display(
    key_df.loc[duplicate_mask]
    .sort_values(flight_key_columns)
    .head(10)
)


## 5. Data Quality Findings

### 5.1 Missing Labels

The January 2023 dataset contains 538,837 flight records.

Flight outcomes:

- Completed flights: 527,197
- Cancelled flights: 10,295
- Diverted flights: 1,345

All completed flights have valid departure and arrival delay labels.

Departure delay labels are missing for 9,982 cancelled flights, while 313 cancelled flights retain departure delay labels.

Arrival delay labels are missing for all cancelled and diverted flights.

### 5.2 Label Consistency

Both binary delay indicators were validated against the corresponding delay duration fields.

Results:

- Departure label mismatches: 0
- Arrival label mismatches: 0
- Invalid binary label values: 0

### 5.3 Duplicate Audit

No repeated candidate flight keys were found.

The candidate key consists of:

- FlightDate
- Reporting_Airline
- Flight_Number_Reporting_Airline
- Origin
- Dest
- CRSDepTime

This key will be revalidated when additional months are ingested.

### 5.4 Initial Modeling Population

For the January 2023 pilot:

- Six-airport flight records: 115,330
- Eligible completed flights: 112,548

The initial modeling cohort excludes cancelled and diverted flights and requires both delay labels to be valid.

### 5.5 Data Cleaning Decisions

1. Preserve original raw BTS records.
2. Restrict modeling data to the selected origin airports.
3. Exclude cancelled and diverted flights from the initial joint modeling cohort.
4. Never replace missing target labels with zero.
5. Validate binary labels and candidate flight keys.
6. Audit missing scheduled features before removing records.
7. Keep flight outcome fields separate from predictive features.

### 5.6 Remaining Limitations

The current audit covers January 2023 only.

The initial cohort is conditional on completed flights, so the models cannot yet provide unconditional delay risk estimates for every booked flight.

BTS retrospective schedules also require further point-in-time availability analysis before production use.


In [ ]:
# Step 2.3.1 - Validate initial cleaning rules

ORIGIN_AIRPORTS = {
    "ATL", "DFW", "DEN", "ORD", "LAX", "SFO"
}

# 1. Define the initial cohort.
in_scope = df["Origin"].isin(ORIGIN_AIRPORTS)

completed = (
    df["Cancelled"].eq(0)
    & df["Diverted"].eq(0)
)

valid_labels = (
    df["DepDel15"].isin([0, 1])
    & df["ArrDel15"].isin([0, 1])
)

eligible = in_scope & completed & valid_labels

# 2. Summarize sample selection.
print("=== Sample Selection ===")

print("Raw records:", len(df))
print("Six-airport records:", int(in_scope.sum()))
print("Completed six-airport flights:",
      int((in_scope & completed).sum()))
print("Eligible modeling records:",
      int(eligible.sum()))

print("\nExcluded from initial cohort:",
      int(in_scope.sum() - eligible.sum()))

# 3. Check missing predictor candidates.
schedule_features = [
    "FlightDate",
    "Reporting_Airline",
    "Origin",
    "Dest",
    "CRSDepTime",
    "CRSArrTime",
    "CRSElapsedTime",
    "Distance",
]

print("\n=== Missing Scheduled Features ===")

missing = df.loc[eligible, schedule_features].isna().sum()
display(missing.to_frame("Missing_Count"))

# 4. Validate target values.
assert df.loc[eligible, "DepDel15"].isin([0, 1]).all()
assert df.loc[eligible, "ArrDel15"].isin([0, 1]).all()

print("\nInitial cleaning rules validated!")


## 6. Reusable Data Validation

### Objective

Convert the exploratory data quality checks into a reusable Python validation module.

**Implementation:** `src/flight_delay/data/validate.py`

### Validation Rules

1. Verify that all required BTS fields exist.
2. Check cancellation and diversion status values.
3. Validate binary delay labels.
4. Verify consistency between delay minutes and binary labels.
5. Require valid labels for completed flights.
6. Check baseline feature completeness in the eligible modeling population.

### Engineering Design

The validation function does not modify input data.

Instead, it raises explicit errors when validation rules fail and returns summary counts when the checks pass.

This fail-fast approach helps prevent invalid data from entering downstream ETL and model training pipelines.

### Pilot Reference Results

- Raw records: 538,837
- Six-airport records: 115,330
- Eligible completed flights: 112,548

These results were obtained during the January 2023 exploratory audit and will be compared with the reusable validation module.


In [ ]:
from flight_delay.data.validate import validate_bts_data

# Reuse the DataFrame loaded during the initial audit.
validation_results = validate_bts_data(df)

print("=== BTS Data Validation Results ===")

for key, value in validation_results.items():
    print(f"{key}: {value:,}")

print("\nAll validation checks passed!")


## 7. Automated Data Validation Tests

### Objective

Verify that the reusable BTS validation module correctly accepts valid data and rejects invalid records.

### Implementation

- Validation module: `src/flight_delay/data/validate.py`
- Unit tests: `tests/test_data_validation.py`
- Testing framework: pytest

### Test Coverage

The tests cover:

1. Valid BTS data acceptance
2. Missing required columns
3. Invalid binary labels
4. Inconsistent delay labels
5. Missing labels in completed flights
6. Missing baseline features
7. Invalid flight status indicators
8. Inconsistent missingness between labels and delay durations

### Testing Strategy

Synthetic test data is used to verify expected behavior without loading the entire BTS dataset.

Both positive and negative test cases are included.

### Limitations

These tests verify the current validation rules but do not yet cover candidate-key uniqueness, cross-month consistency, historical feature availability, or end-to-end pipeline behavior.

Additional tests will be implemented as the ETL pipeline evolves.



### Testing Strategy: Unit vs Integration Tests

The project uses multiple levels of validation.

**Unit Tests**

Small synthetic datasets are used to verify validation logic, including error handling, boundary cases, and invalid inputs.

**Integration Tests**

The January 2023 BTS dataset is used to verify that the validation module works with actual flight records.

Reference results:

- Raw records: 538,837
- Six-airport records: 115,330
- Eligible records: 112,548

**Data Quality Checks**

The production ETL pipeline will perform schema, label, missingness, and duplicate checks for each downloaded month.

Integration tests requiring local raw data are separated from fast unit tests because large datasets are not committed to Git.



## 8. Data Preprocessing Pipeline

### Objective

Implement a reusable preprocessing function to create the initial modeling cohort from validated BTS records.

### Implementation

`src/flight_delay/data/preprocess.py`

### Preprocessing Rules

1. Validate the input dataset before transformation.
2. Retain flights departing from ATL, DFW, DEN, ORD, LAX, or SFO.
3. Exclude cancelled and diverted flights.
4. Require valid `DepDel15` and `ArrDel15` labels.
5. Convert `FlightDate` to datetime.
6. Convert binary target labels to integer types.
7. Preserve the original input DataFrame.

### Engineering Considerations

The preprocessing function returns a new DataFrame and does not modify raw data.

The cleaned dataset retains flight outcome columns for auditing and evaluation. These fields must be excluded from the model feature matrix.

The initial modeling cohort is conditioned on completed flights and does not represent unconditional delay risk across all scheduled flights.

### Expected Pilot Results

- Raw flight records: 538,837
- Six-airport records: 115,330
- Eligible completed flights: 112,548

The preprocessing output should match the previously validated cohort count.



## 9. Preprocessing Unit Tests

### Objective

Verify that the preprocessing pipeline correctly transforms validated BTS data into the initial modeling cohort.

### Implementation

- Module: `src/flight_delay/data/preprocess.py`
- Tests: `tests/test_preprocess.py`
- Framework: pytest

### Test Cases

1. **Cohort Filtering:** Retain completed flights from the selected origin airports.
2. **Data Type Validation:** Convert dates and binary labels to the expected types.
3. **Input Preservation:** Ensure preprocessing does not modify the original DataFrame.
4. **Idempotency:** Verify repeated preprocessing produces identical results.
5. **Invalid Input Handling:** Reject inconsistent delay labels before transformation.

### Testing Strategy

Synthetic flight records are used to test normal, cancelled, diverted, and out-of-scope flights.

These unit tests focus on transformation correctness and error handling.

Real BTS records are validated separately through notebook-based checks and integration testing.

### Limitations

The current tests do not cover multi-month ingestion, Parquet persistence, distributed processing, or point-in-time feature availability.


### Results

- Preprocessing unit tests: 5/5 passed
- Ruff: All checks passed


## 10. Parquet Storage and Verification

### Objective

Persist the validated and preprocessed BTS flight records in Parquet format for downstream feature engineering and model training.

### Implementation

`src/flight_delay/data/storage.py`

### Storage Strategy

- Raw BTS ZIP files are preserved without modification.
- Cleaned records are stored in `data/processed/`.
- Parquet is used for efficient columnar storage.
- Snappy compression is applied.
- The DataFrame index is excluded from storage.
- Temporary-file writing and atomic replacement reduce the risk of partially written output files.

### Data Verification

After writing, the Parquet file will be loaded again to verify:

1. Row and column counts.
2. Column names and ordering.
3. Data types.
4. Departure and arrival target distributions.
5. Equality of the original cleaned DataFrame and the reloaded DataFrame.

### Data Leakage Considerations

Processed data retains historical outcome columns for auditing and model evaluation.

These outcome fields are not approved predictive features and must be excluded when constructing the model feature matrix.

### Version Control

Generated Parquet files are stored locally and excluded from GitHub through `.gitignore`.


In [ ]:
from flight_delay.data.preprocess import prepare_modeling_cohort

# df is the raw January 2023 DataFrame
cleaned_df = prepare_modeling_cohort(df)

print("Cleaned shape:", cleaned_df.shape)
print("Cleaned data ready for Parquet storage!")

In [ ]:
from pathlib import Path

import pandas as pd
from pandas.testing import assert_frame_equal

from flight_delay.data.storage import (
    load_parquet,
    save_parquet,
)

# Locate the project root.
current = Path.cwd().resolve()

ROOT = next(
    (
        path
        for path in [current, *current.parents]
        if (path / "pyproject.toml").exists()
    ),
    None,
)

assert ROOT is not None, "Project root not found."

# cleaned_df was created by prepare_modeling_cohort().
assert "cleaned_df" in globals(), (
    "Run the preprocessing cell first."
)

# Save the January 2023 modeling cohort.
output_path = (
    ROOT / "data" / "processed"
    / "bts_2023_01_completed.parquet"
)

saved_path = save_parquet(cleaned_df, output_path)

print("Saved:", saved_path)
print(
    "File size (MB):",
    round(saved_path.stat().st_size / 1024**2, 2),
)

# Reload the Parquet file.
reloaded_df = load_parquet(saved_path)

print("\n=== Data Verification ===")

print("Original shape:", cleaned_df.shape)
print("Reloaded shape:", reloaded_df.shape)

# Verify full DataFrame equality.
assert_frame_equal(
    cleaned_df,
    reloaded_df,
    check_dtype=True,
)

# Verify expected pilot record count.
assert len(reloaded_df) == 112548

# Verify targets.
for target in ["DepDel15", "ArrDel15"]:
    print(f"\n{target} distribution:")
    print(reloaded_df[target].value_counts().sort_index())

print("\nParquet round-trip verification passed!")


## 11. Parquet Storage Testing

### Objective

Verify the correctness and reliability of the reusable Parquet storage module.

### Implementation

- Module: `src/flight_delay/data/storage.py`
- Tests: `tests/test_storage.py`
- Framework: pytest

### Test Cases

1. **Round-trip Integrity:** Verify that saving and loading preserves DataFrame content and data types.
2. **Directory Creation:** Verify that missing output directories are created.
3. **Idempotent Replacement:** Verify that repeated saves do not append duplicate records.
4. **Invalid Output Format:** Reject non-Parquet file extensions.
5. **Missing File Handling:** Raise an explicit error when the input file does not exist.

### Testing Strategy

Storage tests use small synthetic DataFrames and pytest temporary directories.

This isolates file-system tests from the real BTS datasets.

### Pilot Dataset Verification

The January 2023 cleaned dataset was successfully saved and reloaded.

- Records: 112,548
- Columns: 14
- Compressed file size: approximately 0.76 MB
- DataFrame round-trip equality: Passed

### Limitations

The current storage implementation supports atomic replacement of individual local files.

It does not yet provide concurrent-write coordination, dataset versioning, or distributed storage.



## 12. End-to-End Monthly ETL Pipeline

### Objective

Integrate the reusable data ingestion, validation, preprocessing, and storage components into a single executable monthly ETL pipeline.

### Implementation

`src/flight_delay/data/pipeline.py`

### Pipeline Workflow

1. Locate the monthly BTS ZIP file.
2. Read required columns from the raw CSV.
3. Verify that all flight dates belong to the requested month.
4. Validate source data quality.
5. Prepare the completed-flight modeling cohort.
6. Save the cleaned data to Parquet.
7. Reload and verify the persisted dataset.
8. Return a structured execution summary.

### Command-Line Execution

From the project root:

```bash
python -m flight_delay.data.pipeline --year 2023 --month 1
```

### Engineering Decisions

**Parameterization**

The pipeline accepts year, month, input directory, and output directory as arguments.

**Modularity**

The pipeline reuses existing validation, preprocessing, and storage functions instead of duplicating their logic.

**Fail-Fast Validation**

Invalid source data stops the pipeline before the processed output is replaced.

**Reproducibility**

The same input data and transformation rules should produce equivalent processed data.

**Logging**

The pipeline records key execution stages using Python's logging module.

### Scope and Limitations

This initial implementation processes one local BTS ZIP file at a time.

Automated downloads, multi-month orchestration, distributed processing, and persistent run metadata are planned for later phases.



## 13. Monthly ETL Pipeline Integration Testing

### Objective

Verify that the monthly ETL pipeline correctly processes BTS-style ZIP files and rejects invalid inputs.

### Implementation

- Pipeline: `src/flight_delay/data/pipeline.py`
- Tests: `tests/integration/test_pipeline.py`
- Framework: pytest

### Test Scenarios

1. Successful end-to-end execution
2. Repeatable processing without duplicate appends
3. Invalid month parameters
4. Missing source ZIP files
5. Flight dates outside the requested month
6. Inconsistent delay labels

### Testing Strategy

Synthetic BTS-style ZIP files are generated in temporary directories.

Tests invoke the real pipeline, including CSV loading, validation, preprocessing, Parquet writing, and output verification.

The January 2023 real BTS dataset is tested separately.

### Engineering Value

Integration tests help detect failures that individual module-level tests might miss.

They also establish a regression testing foundation for future multi-month ingestion.

### Remaining Limitations

- Automated monthly downloads are not yet implemented.
- Pipeline tests do not cover every possible malformed CSV or ZIP.
- Cross-month duplicate detection is not implemented.
- Historical feature availability and data leakage checks require further development.


## 14. Flight Identification and Duplicate Audit

### Objective

Preserve flight identification information and audit potential duplicate records in the processed dataset.

### Candidate Flight Key

- FlightDate
- Reporting_Airline
- Flight_Number_Reporting_Airline
- Origin
- Dest
- CRSDepTime

### Validation Strategy

The validation module reports:

1. Records with incomplete candidate keys.
2. Records sharing the same complete candidate key.

Candidate keys are used for data quality auditing and are not assumed to be guaranteed unique identifiers.

Potential duplicates are investigated rather than automatically deleted.

### Feature Engineering Consideration

Flight numbers are preserved for record tracing and auditing but are excluded from the initial model feature allowlist.

### Limitations

Candidate-key uniqueness must be re-evaluated as additional months are ingested.



## 15. Reusable BTS Data Downloader

### Objective

Implement a reusable monthly data downloader for the BTS Reporting Carrier On-Time Performance dataset.

### Implementation

`src/flight_delay/data/download_bts.py`

### Design Decisions

**Parameterized Downloads**

The downloader constructs the official BTS file URL from the requested year and month.

**Caching**

Valid existing files are reused to avoid unnecessary downloads.

**Integrity Checks**

Downloaded archives are checked for ZIP validity, a nonempty CSV, and archive integrity.

**Atomic File Replacement**

New data is written to a temporary file and validated before replacing the destination file.

### Reproducibility

Original BTS ZIP files are retained in `data/raw/` without modification.

Source file checksums and download metadata will be recorded in a later data manifest.

### Limitations

This component validates archive integrity, not the semantic correctness of BTS flight records.

Schema and business-rule validation are performed by the ETL pipeline.



## 16. BTS Downloader Testing

### Objective

Validate the reliability of the reusable BTS downloader under normal and failure scenarios.

### Implementation

- Module: `src/flight_delay/data/download_bts.py`
- Tests: `tests/test_download_bts.py`
- Framework: pytest

### Test Coverage

The tests verify:

1. Correct BTS filename generation
2. Invalid year and month handling
3. ZIP archive integrity checks
4. Missing, multiple, and empty CSV files
5. Cache reuse without network access
6. Successful downloads
7. Recovery from corrupted cached files
8. Network failure handling
9. Temporary file cleanup after failed downloads

### Testing Strategy

Network requests are mocked using pytest's `monkeypatch` fixture.

Synthetic ZIP files are created in memory and stored in temporary directories during testing.

This allows deterministic tests without depending on BTS server availability.

### Limitations

The tests verify downloader behavior but do not guarantee live BTS server availability.

The downloader does not yet implement automatic retries, exponential backoff, or download manifests.

These features may be added when scaling to multi-month ingestion.


In [ ]:
import json
from pathlib import Path

# Locate project root.
current = Path.cwd().resolve()
ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "pyproject.toml").exists()
    ),
    None,
)

assert ROOT is not None

summary_path = (
    ROOT / "data" / "interim"
    / "batch_2023-01_to_2023-02_summary.json"
)

summary = json.loads(
    summary_path.read_text(encoding="utf-8")
)

print("=== Batch ETL Results ===")
print("Requested:", summary["requested_months"])
print("Attempted:", summary["attempted_months"])
print("Succeeded:", summary["succeeded"])
print("Failed:", summary["failed"])

for record in summary["results"]:
    print(
        record["period"],
        record["status"],
        record.get("eligible_records"),
    )

assert summary["succeeded"] == 2
assert summary["failed"] == 0

print("\nTwo-month batch validation passed!")


## 17. Multi-Month BTS ETL Orchestration

### Objective

Extend the monthly BTS processing pipeline to support a configurable range of months.

### Implementation

`src/flight_delay/data/batch_pipeline.py`

### Pipeline Design

The batch runner reuses the existing downloader and monthly ETL modules.

For each requested month:

1. Download or reuse the BTS ZIP.
2. Validate the source data.
3. Prepare the initial modeling cohort.
4. Save the processed Parquet.
5. Verify the output.
6. Record execution status and row counts.

### Engineering Decisions

**Parameterized Date Ranges**

The pipeline supports inclusive start and end months.

**Dry-Run Support**

The requested processing scope can be inspected without downloading or modifying data.

**Failure Reporting**

Failures are recorded in a JSON summary. Failed batches return an error rather than silently reporting success.

**Repeatability**

Existing raw ZIP files are reused when valid, and processed Parquet files are replaced rather than appended.

### Pilot Experiment

The first multi-month test covers January–February 2023.

The batch output and record counts will be verified before extending to the full 2023–2025 period.

### Limitations

- No parallel processing yet
- No automatic download retries
- No input file checksum manifest yet
- No incremental change detection
- No distributed processing yet

These features will be considered after validating the sequential pipeline.



## 18. Batch ETL Verification

### Objective

Validate multi-month BTS data ingestion and establish automated regression tests for batch orchestration.

### January–February 2023 Results

| Metric | January | February |
|---|---:|---:|
| Raw records | 538,837 | 502,749 |
| Eligible records | 112,548 | 105,347 |
| ETL status | Success | Success |

**Combined results:**
- Total raw records: 1,041,586
- Total eligible records: 217,895
- Successful months: 2
- Failed months: 0

### Automated Batch Testing

Implementation: `tests/test_batch_pipeline.py`

Test scenarios include:
- Cross-year month iteration
- Invalid date ranges
- Successful batch execution
- Offline processing
- Fail-fast behavior
- Continue-on-error behavior
- Dry-run execution

### Next Steps

1. Validate batch orchestration tests.
2. Compare schemas and label distributions across months.
3. Extend ingestion to 2023–2025.
4. Generate a comprehensive data quality report.

The initial modeling population remains restricted to completed flights from the six selected origin airports.


In [ ]:
import json

summary_path = (
    ROOT / "data" / "interim"
    / "batch_2023-01_to_2023-02_summary.json"
)

summary = json.loads(summary_path.read_text())

records = summary["results"]

total_raw = sum(r["raw_records"] for r in records)
total_eligible = sum(r["eligible_records"] for r in records)

print("Total raw records:", f"{total_raw:,}")
print("Total eligible records:", f"{total_eligible:,}")
print("Succeeded:", summary["succeeded"])
print("Failed:", summary["failed"])

assert summary["succeeded"] == 2
assert summary["failed"] == 0
assert total_raw == 1_041_586
assert total_eligible == 217_895

print("\nBatch result verification passed!")


## 19. Cross-Month Schema and Data Quality Audit

### Objective

Evaluate data consistency across January and February 2023 before scaling the ETL pipeline to the full 2023–2025 dataset.

### Audit Questions

1. Do the processed datasets have consistent column names and data types?
2. Are required modeling features complete in both months?
3. Are there duplicate candidate flight keys within or across months?
4. Are departure and arrival delay labels valid?
5. How do delay rates and flight volumes differ between months?

### Methodology

The audit uses the processed monthly Parquet files generated by the ETL pipeline.

Data quality is evaluated separately for each month, followed by a cross-month consistency check.

### Important Limitation

The processed datasets contain completed flights from the six selected origin airports.

Therefore, this audit does not measure cancellation or diversion rates for the complete raw monthly datasets.


In [ ]:
import pandas as pd
from pathlib import Path

from flight_delay.data.storage import load_parquet

# Locate repository root.
ROOT = next(
    (
        p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
        if (p / "pyproject.toml").exists()
    ),
    None,
)
assert ROOT is not None

# Load both processed datasets.
monthly_data = {}

for period in ["2023-01", "2023-02"]:
    year, month = period.split("-")

    path = (
        ROOT / "data" / "processed"
        / f"bts_{year}_{month}_completed.parquet"
    )

    monthly_data[period] = load_parquet(path)

# Compare schemas.
jan = monthly_data["2023-01"]
feb = monthly_data["2023-02"]

print("January Shape:", jan.shape)
print("February Shape:", feb.shape)

print("\n=== Schema Comparison ===")

schema_report = pd.DataFrame({
    "Column": jan.columns,
    "January_Dtype": [str(x) for x in jan.dtypes],
    "February_Dtype": [
        str(feb[col].dtype) if col in feb.columns else "MISSING"
        for col in jan.columns
    ],
})

display(schema_report)

assert jan.columns.equals(feb.columns), (
    "Column names or ordering differ between months."
)

assert jan.dtypes.equals(feb.dtypes), (
    "Column data types differ between months."
)

print("\nCross-month schema validation passed!")

In [ ]:
from flight_delay.data.validate import (
    CANDIDATE_KEY_COLUMNS,
    validate_bts_data,
)
from flight_delay.features.schema import BASELINE_FEATURES

quality_results = []

for period, month_df in monthly_data.items():

    # Reuse existing validation rules.
    audit = validate_bts_data(month_df)

    # Check date range.
    expected_month = pd.Period(period, freq="M")
    actual_months = pd.to_datetime(
        month_df["FlightDate"]
    ).dt.to_period("M")

    assert actual_months.eq(expected_month).all()

    # Check duplicate candidate flight keys.
    duplicates = month_df.duplicated(
        subset=CANDIDATE_KEY_COLUMNS,
        keep=False,
    ).sum()

    # Check missing baseline predictors.
    missing_features = (
        month_df[list(BASELINE_FEATURES)]
        .isna()
        .sum()
        .sum()
    )

    quality_results.append({
        "Month": period,
        "Records": len(month_df),
        "Departure_Delay_Rate": round(
            month_df["DepDel15"].mean() * 100, 2
        ),
        "Arrival_Delay_Rate": round(
            month_df["ArrDel15"].mean() * 100, 2
        ),
        "Missing_Features": int(missing_features),
        "Incomplete_Keys": audit[
            "incomplete_candidate_key_records"
        ],
        "Duplicate_Keys": int(duplicates),
        "Airports": month_df["Origin"].nunique(),
    })

quality_report = pd.DataFrame(quality_results)

print("=== Monthly Data Quality Report ===")
display(quality_report)

# Check duplicate keys after combining months.
combined = pd.concat(
    monthly_data.values(),
    ignore_index=True,
)

cross_month_duplicates = combined.duplicated(
    subset=CANDIDATE_KEY_COLUMNS,
    keep=False,
).sum()

print("Combined records:", len(combined))
print("Repeated candidate keys:", int(cross_month_duplicates))

assert quality_report["Missing_Features"].eq(0).all()
assert quality_report["Incomplete_Keys"].eq(0).all()
assert quality_report["Duplicate_Keys"].eq(0).all()
assert cross_month_duplicates == 0

print("\nCross-month data quality checks passed!")


## 20. Data Provenance and Manifest Generation

### Objective

Record file-level provenance and schema metadata for each processed BTS month.

### Implementation

`src/flight_delay/data/manifest.py`

### Metadata

Each monthly manifest includes:

- Source ZIP filename and SHA-256 checksum
- Processed Parquet filename and SHA-256 checksum
- Dataset version
- Schema fingerprint
- Row and column counts
- Departure and arrival delay rates
- Candidate flight key quality metrics
- UTC manifest generation timestamp

### Engineering Rationale

Checksums detect changes in source and processed files.

Schema fingerprints help identify structural inconsistencies across months.

Dataset versioning documents the intended processing contract.

### Limitations

The current manifest is generated from existing local files.

Future improvements include recording the Git commit, dependency versions, configuration hashes, and unique pipeline run identifiers.



## 21. May 2023 Data Quality Exception

During full-year ingestion, the May 2023 BTS dataset contained one completed flight with a missing arrival delay outcome.

The affected flight was YX 5612, LGA to BNA, on May 21, 2023.

Because LGA was outside the six selected origin airports, this record did not affect the modeling cohort.

The validation logic was updated to audit missing labels among completed flights while requiring complete labels for all modeling records.

No target labels were imputed, and no records were automatically deduplicated.

After the fix:
- Raw records: 579,958
- Eligible modeling records: 126,159
- Completed flights with missing labels: 1
- In-scope completed flights with missing labels: 0

The updated ETL pipeline completed successfully.


In [ ]:
import pandas as pd

from flight_delay.data.storage import load_parquet
from flight_delay.data.validate import CANDIDATE_KEY_COLUMNS
from flight_delay.features.schema import BASELINE_FEATURES

monthly_frames = []
monthly_results = []

for month in range(1, 13):
    period = f"2023-{month:02d}"

    path = (
        ROOT / "data" / "processed"
        / f"bts_2023_{month:02d}_completed.parquet"
    )

    df = load_parquet(path)

    monthly_results.append({
        "Month": period,
        "Records": len(df),
        "DepDelayRate": round(df["DepDel15"].mean() * 100, 2),
        "ArrDelayRate": round(df["ArrDel15"].mean() * 100, 2),
        "MissingFeatures": int(
            df[list(BASELINE_FEATURES)].isna().sum().sum()
        ),
        "IncompleteKeys": int(
            df[CANDIDATE_KEY_COLUMNS].isna().any(axis=1).sum()
        ),
    })

    monthly_frames.append(df)

monthly_report = pd.DataFrame(monthly_results)
all_2023 = pd.concat(monthly_frames, ignore_index=True)

duplicate_keys = int(
    all_2023.duplicated(
        subset=CANDIDATE_KEY_COLUMNS,
        keep=False,
    ).sum()
)

print("=== 2023 Annual Summary ===")
print("Total eligible flights:", f"{len(all_2023):,}")
print(
    "Departure delay rate:",
    f"{all_2023['DepDel15'].mean():.2%}",
)
print(
    "Arrival delay rate:",
    f"{all_2023['ArrDel15'].mean():.2%}",
)
print("Cross-month duplicate keys:", duplicate_keys)
print(
    "Missing baseline features:",
    monthly_report["MissingFeatures"].sum(),
)

assert len(all_2023) == 1_478_412
assert duplicate_keys == 0
assert monthly_report["MissingFeatures"].eq(0).all()
assert monthly_report["IncompleteKeys"].eq(0).all()

display(monthly_report)

print("\n2023 annual quality audit passed!")

### 2023 Historical Data Ingestion Results

The complete 2023 BTS Reporting Carrier On-Time Performance dataset was successfully ingested and processed.

**Dataset Summary**

- Period: January–December 2023
- Raw flight records: 6,847,899
- Eligible modeling records: 1,478,412
- Origin airports: ATL, DFW, DEN, ORD, LAX, SFO
- Monthly Parquet files: 12
- Monthly data manifests: 12
- Cross-month schema fingerprints: Consistent

**Pipeline Capabilities**

The ingestion system supports:

- Automated BTS ZIP downloads
- Local file caching
- Source archive validation
- Data quality checks
- Modeling cohort filtering
- Compressed Parquet storage
- File-level SHA-256 checksums
- Monthly manifests
- Batch execution and failure reporting

**Modeling Considerations**

The dataset supports two separate binary classification tasks: departure delay and arrival delay prediction.

Both tasks use the same completed-flight cohort.

Temporal train/validation/test splits and point-in-time feature availability constraints will be enforced during model development.


### 2023 Annual Audit — Findings and Discussion

The 2023 modeling dataset contains 1,478,412 eligible flight records from six selected origin airports.

The overall departure delay rate is 22.01%, while the arrival delay rate is 21.96%.

All required baseline features are complete, and no duplicate candidate flight keys were detected.

**Temporal Variation**

Monthly delay rates vary considerably throughout the year.

Departure delays peaked in July (30.87%) and reached their lowest level in November (13.89%), representing a difference of 16.98 percentage points.

Departure and arrival delay rates followed similar monthly patterns.

**Modeling Implications**

These findings support evaluating models with chronological data splits rather than random train-test splits.

Calendar-related features may provide useful predictive information.

Historical operational features must be constructed using only information available before the T-24h prediction timestamp.

The observed monthly variation is descriptive and does not establish its underlying causes.

**Next Step**

Process and audit the 2024 dataset to examine whether these temporal patterns persist across years.



## 22. Full-Year 2024 Data Quality Audit

### Objective

Validate the complete 2024 dataset and confirm consistency with the 2023 training data.

### Data Scope

- Period: January–December 2024
- Origin airports: ATL, DFW, DEN, ORD, LAX, SFO
- Modeling cohort: Completed, non-diverted flights with both delay labels available

### Quality Checks

1. All 12 monthly Parquet files exist.
2. Monthly schemas are consistent.
3. Flight dates match the expected months.
4. Baseline features have no missing values.
5. Candidate flight keys are complete and unique.
6. Both prediction targets contain valid binary labels.

### Temporal Evaluation Design

- January–August: Model validation
- September–October: Probability calibration
- November–December: Decision threshold selection

The 2025 dataset remains reserved for final out-of-time testing.

### Status

The 2024 monthly ETL completed successfully. The full-year data quality audit is performed separately below.


In [ ]:
import pandas as pd
from pathlib import Path

from flight_delay.data.storage import load_parquet
from flight_delay.data.validate import (
    CANDIDATE_KEY_COLUMNS,
    validate_bts_data,
)
from flight_delay.features.schema import BASELINE_FEATURES

AUDIT_YEAR = 2024

# Locate repository root.
current = Path.cwd().resolve()
ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "pyproject.toml").exists()
    ),
    None,
)
assert ROOT is not None

monthly_results = []
key_frames = []
reference_schema = None

for month in range(1, 13):
    period = f"{AUDIT_YEAR}-{month:02d}"

    path = (
        ROOT / "data" / "processed"
        / f"bts_{AUDIT_YEAR}_{month:02d}_completed.parquet"
    )

    assert path.is_file(), f"Missing file: {path}"

    df = load_parquet(path)

    # Check schema consistency.
    schema = tuple(
        (column, str(dtype))
        for column, dtype in df.dtypes.items()
    )

    if reference_schema is None:
        reference_schema = schema
    else:
        assert schema == reference_schema, (
            f"Schema mismatch: {period}"
        )

    # Verify flight dates.
    dates = pd.to_datetime(df["FlightDate"])

    assert dates.dt.to_period("M").eq(
        pd.Period(period)
    ).all(), f"Incorrect dates: {period}"

    # Reuse existing data validation rules.
    audit = validate_bts_data(df)

    assert audit["eligible_records"] == len(df)

    # Quality metrics.
    missing_features = int(
        df[list(BASELINE_FEATURES)].isna().sum().sum()
    )

    incomplete_keys = int(
        df[CANDIDATE_KEY_COLUMNS]
        .isna()
        .any(axis=1)
        .sum()
    )

    duplicate_keys = int(
        df.duplicated(
            subset=CANDIDATE_KEY_COLUMNS,
            keep=False,
        ).sum()
    )

    monthly_results.append({
        "Month": period,
        "MonthNumber": month,
        "Records": len(df),
        "DepPositive": int(df["DepDel15"].sum()),
        "ArrPositive": int(df["ArrDel15"].sum()),
        "DepDelayRate": round(
            df["DepDel15"].mean() * 100, 2
        ),
        "ArrDelayRate": round(
            df["ArrDel15"].mean() * 100, 2
        ),
        "MissingFeatures": missing_features,
        "IncompleteKeys": incomplete_keys,
        "DuplicateKeys": duplicate_keys,
    })

    # Only retain candidate keys for the annual duplicate audit.
    key_frames.append(df[CANDIDATE_KEY_COLUMNS].copy())

report_2024 = pd.DataFrame(monthly_results)

all_keys = pd.concat(key_frames, ignore_index=True)

annual_duplicates = int(
    all_keys.duplicated(
        subset=CANDIDATE_KEY_COLUMNS,
        keep=False,
    ).sum()
)

total_records = int(report_2024["Records"].sum())

dep_rate = (
    report_2024["DepPositive"].sum() / total_records
)

arr_rate = (
    report_2024["ArrPositive"].sum() / total_records
)

print("=== 2024 Annual Summary ===")
print("Total eligible flights:", f"{total_records:,}")
print("Departure delay rate:", f"{dep_rate:.2%}")
print("Arrival delay rate:", f"{arr_rate:.2%}")
print("Cross-month duplicate keys:", annual_duplicates)
print(
    "Missing baseline features:",
    report_2024["MissingFeatures"].sum(),
)

display(
    report_2024.drop(
        columns=["MonthNumber", "DepPositive", "ArrPositive"]
    )
)

assert total_records == 1_549_111
assert report_2024["MissingFeatures"].eq(0).all()
assert report_2024["IncompleteKeys"].eq(0).all()
assert report_2024["DuplicateKeys"].eq(0).all()
assert annual_duplicates == 0

print("\n2024 annual quality audit passed!")

In [ ]:
evaluation_splits = {
    "Validation (Jan-Aug)": (1, 8),
    "Calibration (Sep-Oct)": (9, 10),
    "Threshold (Nov-Dec)": (11, 12),
}

split_results = []

for name, (start_month, end_month) in evaluation_splits.items():
    subset = report_2024[
        report_2024["MonthNumber"].between(
            start_month, end_month
        )
    ]

    n = int(subset["Records"].sum())

    split_results.append({
        "Split": name,
        "Records": n,
        "DepDelayRate": round(
            subset["DepPositive"].sum() / n * 100, 2
        ),
        "ArrDelayRate": round(
            subset["ArrPositive"].sum() / n * 100, 2
        ),
    })

split_report = pd.DataFrame(split_results)

display(split_report)

assert split_report["Records"].sum() == 1_549_111

print("\nTemporal split verification passed!")


## 23. 2025 Out-of-Time Test Data Quality Audit

### Objective

Validate the structural integrity of the reserved 2025 test dataset without using its target distributions to inform model development.

### Validation Scope

- All 12 monthly Parquet files must exist.
- Schema must match the historical training data.
- Flight dates must match each monthly partition.
- Baseline predictors must be complete.
- Candidate flight keys must be complete and unique.
- Existing data validation rules must pass.

### Test Isolation Policy

The 2025 dataset is reserved for final out-of-time evaluation.

Its label distributions and predictive relationships will not be used to guide feature selection, hyperparameter tuning, calibration, or threshold optimization.


In [ ]:
import pandas as pd

from flight_delay.data.storage import load_parquet
from flight_delay.data.validate import (
    CANDIDATE_KEY_COLUMNS,
    validate_bts_data,
)
from flight_delay.features.schema import BASELINE_FEATURES

reference = load_parquet(
    ROOT / "data/processed/bts_2023_01_completed.parquet"
)

reference_schema = tuple(
    (col, str(dtype))
    for col, dtype in reference.dtypes.items()
)

results = []
key_frames = []

for month in range(1, 13):
    period = f"2025-{month:02d}"

    path = (
        ROOT / "data/processed"
        / f"bts_2025_{month:02d}_completed.parquet"
    )

    assert path.is_file(), f"Missing: {path}"

    df = load_parquet(path)

    assert tuple(
        (col, str(dtype))
        for col, dtype in df.dtypes.items()
    ) == reference_schema

    assert pd.to_datetime(
        df["FlightDate"]
    ).dt.to_period("M").eq(pd.Period(period)).all()

    audit = validate_bts_data(df)

    assert audit["eligible_records"] == len(df)

    missing = int(
        df[list(BASELINE_FEATURES)].isna().sum().sum()
    )

    incomplete = int(
        df[CANDIDATE_KEY_COLUMNS]
        .isna().any(axis=1).sum()
    )

    results.append({
        "Month": period,
        "Records": len(df),
        "MissingFeatures": missing,
        "IncompleteKeys": incomplete,
    })

    key_frames.append(df[CANDIDATE_KEY_COLUMNS])

report_2025 = pd.DataFrame(results)

all_keys = pd.concat(key_frames, ignore_index=True)

duplicate_keys = int(
    all_keys.duplicated(
        subset=CANDIDATE_KEY_COLUMNS,
        keep=False,
    ).sum()
)

display(report_2025)

print("Total records:", f"{report_2025['Records'].sum():,}")
print("Duplicate candidate keys:", duplicate_keys)

assert report_2025["Records"].sum() == 1_581_499
assert report_2025["MissingFeatures"].eq(0).all()
assert report_2025["IncompleteKeys"].eq(0).all()
assert duplicate_keys == 0

print("\n2025 structural audit passed!")